# Forward Curve Evaluation

This notebook evaluates forward curve recipes (PCHIP and Kalman) using:
- **Put-call parity**: Compare fitted forwards to options-implied forwards
- **Pillar fit quality**: WMAE between fitted curve and pillar observations
- **Leave-one-expiry-out (LOEO)**: Cross-validation by excluding each pillar

We test on two scenarios:
- **(a)** Full month with binning: 2025-09-01 to 2025-10-01 with 5m bins
- **(b)** Single day with option timestamps: 2025-09-02 using OPTIONS data timestamps

## Key Design

All evaluations use `assign_forwards()` from `okx.recipes.forwards` to match forward
prices to data. This keeps the pipeline consistent and leverages the caching system.


## Imports

In [2]:
from datetime import date, datetime, timedelta
from functools import partial
from typing import Dict, Optional

import numpy as np
import polars as pl
from tqdm import tqdm

from okx.store import OrderbookStore
from okx.recipes.forwards import build_forwards_pchip, build_forwards_kalman, assign_forwards
from okx.recipes.options import prepare_options
from evaluation.forwards_eval import evaluate_parity, summarize_parity, evaluate_pillar_fit, evaluate_loeo

## Store Configuration

In [3]:
store = OrderbookStore(
    data_root="data/okx",
    manifest_path="data/okx/manifest.sqlite"
)

In [5]:
store.clear_cache()

Cleared all caches


## Evaluation Parameters

In [4]:
# Shared parameters
inst_family = 'BTC-USD'

# Scenario (a): Full month with binning
dates_month = [date(2025, 9, 1) + timedelta(days=i) for i in range(30)]  # Sep 1-30
binning_month = '5m'

# Scenario (b): Single day with option timestamps
dates_day = [date(2025, 9, 2)]
binning_day = None  # Will use unique_times from OPTIONS

# Put-call parity filters
moneyness_spread = 0.05

# Recipe configurations
recipes = {
    'pchip': build_forwards_pchip,
    'kalman': build_forwards_kalman
}

In [ ]:
lf_options = store.get(
    inst_family=inst_family,
    inst_type='OPTION',
    dates=[date(2025, 9, 2), date(2025, 9, 3)],
    depth=0,
    features=['trim', 'strip', 'parse_option'],
    verbose=True,
    benchmark=True
)

[store] Getting BTC-USD/OPTION for 2 dates (depth=0, binning=None, 3 features)
  - applied 'trim'
  [benchmark] trim: 0.192s
  - applied 'strip'
  [benchmark] strip: 0.000s
  - applied 'parse_option_v1'
  [benchmark] parse_option_v1: 5.876s
  [benchmark] Direct build: 6.073s
[store] Getting BTC-USD/OPTION for 2 dates (depth=0, binning=None, 3 features)
  - applied 'trim'
  [benchmark] trim: 0.132s
  - applied 'strip'
  [benchmark] strip: 0.000s
  - applied 'parse_option_v2'
  [benchmark] parse_option_v2: 0.944s
  [benchmark] Direct build: 1.079s


In [5]:
lf_errors = evaluate_pillar_fit(
    store=store,
    dates=dates_month,
    inst_family=inst_family,
    forwards_recipe=recipes['kalman'],
    binning=binning_month,
)

Preparing pillars for 30 dates...
Constructing pillars for BTC-USD with swap and futures using 5m binning
 - Time taken to fetch swap and futures: 0:00:00.065341
 - Time taken to align snapshots: 0:00:00.001846
 - Time taken to concatenate and filter: 0:00:00.016092
 - Time taken to index and sort pillars: 0:00:00.000327
 - Total time taken to prepare pillars: 0:00:00.083606
Assigning kalman forwards to 30 dates using kalman forwards
 - Time taken to collect data: 0:00:00.016538
[store] Getting derived data via recipe 'build_forwards_kalman' for 30 dates
 - Time taken to fetch forwards: 0:00:00.028597
 - Time taken to build forward lookup: 0:00:00.032259


Matching forwards: 100%|██████████| 8639/8639 [00:00<00:00, 58112.03it/s]

 - Time taken to match forwards: 0:00:00.150900, matched 68,248 / 68,248 rows
 - Total time taken to assign forwards: 0:00:00.211756


In [11]:
df_errors = lf_errors.collect()
df_errors = df_errors.filter(pl.col('pillar_idx') != 0)
metrics = df_errors.group_by(['pillar_idx']).agg([
    pl.col('error_bid_bps').mean().alias('mae_bid_bps'),
    pl.col('error_ask_bps').mean().alias('mae_ask_bps'),
    pl.col('error_mid_bps').mean().alias('mae_mid_bps'),
    ((pl.col('F_ask_obs') < pl.col('F_bid_pred')).sum() / pl.len()).alias('spread_above_pct'),
    ((pl.col('F_bid_obs') > pl.col('F_ask_pred')).sum() / pl.len()).alias('spread_below_pct'),
])
metrics.sort('pillar_idx').head()


pillar_idx,mae_bid_bps,mae_ask_bps,mae_mid_bps,spread_above_pct,spread_below_pct
i64,f64,f64,f64,f64,f64
1,2.011165,2.113883,2.058759,0.129876,0.851835
2,2.285127,2.209824,2.241302,0.564301,0.408612
3,4.017288,3.907714,3.957395,0.753444,0.231856
4,2.673521,2.45376,2.55275,0.685612,0.277926
5,1.823463,1.819796,1.821302,0.663503,0.329784


## Scenario (a): Full Month with Binning

Evaluate both recipes on 2025-09-01 to 2025-09-30 with 5-minute binning.

In [ ]:
results_month = {}

for recipe_name, recipe_fn in recipes.items():
    print(f"\n{'='*60}")
    print(f"Evaluating {recipe_name.upper()} on FULL MONTH with {binning_month} binning")
    print(f"{'='*60}\n")
    
    recipe_configured = partial(
        recipe_fn,
        inst_family=inst_family,
        min_time_to_expiry_hours=min_time_to_expiry_hours,
    )
    
    results_month[recipe_name] = {}
    
    # Put-call parity
    print(f"\n--- Put-Call Parity ---")
    df_parity, summary_parity = evaluate_put_call_parity(
        store=store,
        dates=dates_month,
        inst_family=inst_family,
        forwards_recipe=recipe_configured,
        binning=binning_month,
        min_moneyness=moneyness_spread,
        verbose=True,
    )
    results_month[recipe_name]['parity'] = df_parity
    print("\nSummary:")
    print(summary_parity)
    
    # Pillar fit
    print(f"\n--- Pillar Fit Quality ---")
    df_pillar_fit = evaluate_pillar_fit(
        store=store,
        dates=dates_month,
        inst_family=inst_family,
        forwards_recipe=recipe_configured,
        binning=binning_month,
        min_time_to_expiry_hours=min_time_to_expiry_hours,
        verbose=True,
    )
    results_month[recipe_name]['pillar_fit'] = df_pillar_fit
    
    if not df_pillar_fit.is_empty():
        summary_pillar = df_pillar_fit.group_by('side').agg([
            pl.col('wmae_bps').mean().alias('mean_wmae_bps'),
            pl.col('wmae_bps').std().alias('std_wmae_bps'),
            pl.col('max_error_bps').max().alias('max_error_bps'),
        ])
        print("\nSummary:")
        print(summary_pillar)
    
    # LOEO - sample first 3 days only for speed
    print(f"\n--- Leave-One-Expiry-Out (first 3 days) ---")
    df_loeo = evaluate_loeo(
        store=store,
        dates=dates_month[:3],  # First 3 days only
        inst_family=inst_family,
        forwards_recipe=recipe_configured,
        binning=binning_month,
        min_time_to_expiry_hours=min_time_to_expiry_hours,
        verbose=True,
    )
    results_month[recipe_name]['loeo'] = df_loeo
    
    if not df_loeo.is_empty():
        summary_loeo = df_loeo.filter(pl.col('success')).select([
            pl.col('error_mid_bps').mean().alias('mean_error_mid_bps'),
            pl.col('error_mid_bps').std().alias('std_error_mid_bps'),
            pl.col('error_mid_bps').max().alias('max_error_mid_bps'),
        ])
        print("\nSummary:")
        print(summary_loeo)


## Scenario (b): Single Day with Option Timestamps

Evaluate both recipes on 2025-09-02 using unique timestamps from OPTIONS orderbook.


In [ ]:
# Get unique timestamps from OPTIONS orderbook
print("Fetching unique timestamps from OPTIONS orderbook...")
df_options_raw = store.get(
    inst_family=inst_family,
    inst_type='OPTION',
    dates=dates_day,
    depth=0,
    features=['trim', 'strip', 'dedupe'],
    verbose=False,
).collect()

unique_times = df_options_raw['timeMs'].unique().sort().to_list()
print(f"Found {len(unique_times)} unique timestamps")


In [ ]:
results_day = {}

for recipe_name, recipe_fn in recipes.items():
    print(f"\n{'='*60}")
    print(f"Evaluating {recipe_name.upper()} on SINGLE DAY with OPTION TIMESTAMPS")
    print(f"{'='*60}\n")
    
    recipe_configured = partial(
        recipe_fn,
        inst_family=inst_family,
        min_time_to_expiry_hours=min_time_to_expiry_hours,
    )
    
    results_day[recipe_name] = {}
    
    # Put-call parity
    print(f"\n--- Put-Call Parity ---")
    df_parity, summary_parity = evaluate_put_call_parity(
        store=store,
        dates=dates_day,
        inst_family=inst_family,
        forwards_recipe=recipe_configured,
        binning=binning_day,
        min_time_to_expiry_hours=min_time_to_expiry_hours,
        min_moneyness=min_moneyness,
        max_moneyness=max_moneyness,
        verbose=True,
    )
    results_day[recipe_name]['parity'] = df_parity
    print("\nSummary:")
    print(summary_parity)
    
    # Pillar fit
    print(f"\n--- Pillar Fit Quality ---")
    df_pillar_fit = evaluate_pillar_fit(
        store=store,
        dates=dates_day,
        inst_family=inst_family,
        forwards_recipe=recipe_configured,
        binning=binning_day,
        min_time_to_expiry_hours=min_time_to_expiry_hours,
        unique_times=unique_times,
        verbose=True,
    )
    results_day[recipe_name]['pillar_fit'] = df_pillar_fit
    
    if not df_pillar_fit.is_empty():
        summary_pillar = df_pillar_fit.group_by('side').agg([
            pl.col('wmae_bps').mean().alias('mean_wmae_bps'),
            pl.col('wmae_bps').std().alias('std_wmae_bps'),
            pl.col('max_error_bps').max().alias('max_error_bps'),
        ])
        print("\nSummary:")
        print(summary_pillar)
    
    # LOEO - sample every 50th timestamp for speed
    print(f"\n--- Leave-One-Expiry-Out (sampled timestamps) ---")
    unique_times_sampled = unique_times[::50]  # Sample every 50th
    print(f"Using {len(unique_times_sampled)} sampled timestamps")
    
    df_loeo = evaluate_loeo(
        store=store,
        dates=dates_day,
        inst_family=inst_family,
        forwards_recipe=recipe_configured,
        binning=binning_day,
        min_time_to_expiry_hours=min_time_to_expiry_hours,
        unique_times=unique_times_sampled,
        verbose=True,
    )
    results_day[recipe_name]['loeo'] = df_loeo
    
    if not df_loeo.is_empty():
        summary_loeo = df_loeo.filter(pl.col('success')).select([
            pl.col('error_mid_bps').mean().alias('mean_error_mid_bps'),
            pl.col('error_mid_bps').std().alias('std_error_mid_bps'),
            pl.col('error_mid_bps').max().alias('max_error_mid_bps'),
        ])
        print("\nSummary:")
        print(summary_loeo)


## Comparison Across Scenarios

Compare PCHIP vs Kalman performance across both scenarios.


In [ ]:
print("\n" + "="*60)
print("COMPARISON SUMMARY")
print("="*60)

# Put-call parity comparison
print("\n--- Put-Call Parity Errors (bps) ---\n")
comparison_data = []
for scenario, results in [('Month+Binning', results_month), ('Day+Options', results_day)]:
    for recipe_name in recipes.keys():
        df_parity = results[recipe_name]['parity']
        if not df_parity.is_empty():
            comparison_data.append({
                'scenario': scenario,
                'recipe': recipe_name,
                'n_pairs': len(df_parity),
                'error_mid_mean': df_parity['error_mid_bps'].mean(),
                'error_mid_std': df_parity['error_mid_bps'].std(),
            })

if comparison_data:
    print(pl.DataFrame(comparison_data))

# Pillar fit comparison
print("\n--- Pillar Fit WMAE (bps) ---\n")
comparison_data = []
for scenario, results in [('Month+Binning', results_month), ('Day+Options', results_day)]:
    for recipe_name in recipes.keys():
        df_fit = results[recipe_name]['pillar_fit']
        if not df_fit.is_empty():
            for side in ['bid', 'ask', 'mid']:
                side_data = df_fit.filter(pl.col('side') == side)
                if not side_data.is_empty():
                    comparison_data.append({
                        'scenario': scenario,
                        'recipe': recipe_name,
                        'side': side,
                        'mean_wmae': side_data['wmae_bps'].mean(),
                        'max_error': side_data['max_error_bps'].max(),
                    })

if comparison_data:
    print(pl.DataFrame(comparison_data))

# LOEO comparison
print("\n--- LOEO Errors (bps) ---\n")
comparison_data = []
for scenario, results in [('Month+Binning', results_month), ('Day+Options', results_day)]:
    for recipe_name in recipes.keys():
        df_loeo = results[recipe_name]['loeo']
        if not df_loeo.is_empty():
            success_df = df_loeo.filter(pl.col('success'))
            if not success_df.is_empty():
                comparison_data.append({
                    'scenario': scenario,
                    'recipe': recipe_name,
                    'n_tests': len(success_df),
                    'mean_error_mid': success_df['error_mid_bps'].mean(),
                    'max_error_mid': success_df['error_mid_bps'].max(),
                })

if comparison_data:
    print(pl.DataFrame(comparison_data))


---

## Key Takeaways

This evaluation framework provides a comprehensive assessment of forward curve quality:

1. **Put-call parity** validates consistency with options market
2. **Pillar fit** measures how well the curve fits observed forward prices
3. **LOEO** tests out-of-sample prediction performance

The modular design allows easy comparison across:
- Different recipes (PCHIP vs Kalman)
- Different time horizons (full month vs single day)
- Different sampling strategies (binning vs option timestamps)